




# Project: AI Excuse Generator and Assistant


In [3]:
# pip install accelerate
from transformers import T5Tokenizer,T5ForConditionalGeneration,MarianMTModel,MarianTokenizer,pipeline,BartTokenizer,BartForConditionalGeneration,AutoTokenizer,AutoModelForSeq2SeqLM
from PIL import Image, ImageDraw, ImageFont
from datetime import datetime
from collections import Counter
from TTS.api import TTS
import IPython.display as ipd

tts_model = TTS(model_name="tts_models/en/ljspeech/tacotron2-DDC", progress_bar=False, gpu=True)


summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
import os

para_tokenizer = AutoTokenizer.from_pretrained("ramsrigouthamg/t5_paraphraser")
para_model = AutoModelForSeq2SeqLM.from_pretrained("ramsrigouthamg/t5_paraphraser").to("cuda")
flan_tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-large")
flan_model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-large", device_map="auto")
trans_tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-fr")
trans_model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-fr").to("cuda")


history="excuse_history.txt"
fav="favourites.txt"
last=None
last_excuse=None

def create_proof_image(text, filename="proof.png"):
    img = Image.new("RGB", (800, 600), color=(255, 255, 255))
    d = ImageDraw.Draw(img)
    font = ImageFont.load_default()
    d.text((50, 100), text, fill=(0, 0, 0), font=font)
    img.save(filename)
    print(f"Proof image saved as {filename}")

def generate_proof_document(name="Jagath", reason="personal emergency", date=None):
    if date is None:
        date = datetime.now().strftime("%B %d, %Y")
    return f"This is to certify that Mr./Ms. {name} was on {reason} on {date}. Kindly grant leave accordingly."

def paraphrase_text(text):
    input_text = f"paraphrase: {text} </s>"
    encoding = para_tokenizer.encode_plus(input_text, padding="longest", return_tensors="pt").to("cuda")
    generated_ids = para_model.generate(
        input_ids=encoding["input_ids"],
        attention_mask=encoding["attention_mask"],
        max_length=256,
        num_beams=5,
        num_return_sequences=1,
        no_repeat_ngram_size=2,
        early_stopping=True,
    )
    return para_tokenizer.decode(generated_ids[0], skip_special_tokens=True)

def rank_excuses(file_path):
    if os.path.exists(file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            all_excuses = [line.strip() for line in f.readlines()]
        counter = Counter(all_excuses)
        ranked = counter.most_common(5)
        print("\nTop 5 Excuses:")
        for excuse, count in ranked:
            print(f"{excuse} (used {count} times)")
        print()
    else:
        print("No history found for ranking.")

usedexcuse=[]
if os.path.exists(history):
  with open(history,"r",encoding="utf-8")as f:
    usedexcuse=[line.strip()for line in f.readlines()]

print("Bot: Hello! How can i help you ?")
used_excuses=[]
while True:
  input_text=input("You : ")
  if input_text.lower()in['exit','quit','bye']:
    print("Bot : Good bye !")
    break
  elif input_text=="show history":
    print("\n   history    ")
    for excuse in usedexcuse:
      print("-",excuse)
    print()
  elif "save favorites" in input_text.lower():
    if last:
      with open("fav.txt","a",encoding="utf-8") as f:
        f.write(last+"\n")
      print("Bot: Response saved to favorites")
    else:
      print("Bot : No response to save . ")
  elif "translate" in input_text.lower():
    text_to_translate=input_text.split(":",1)[-1].strip()
    tokens=trans_tokenizer(text_to_translate,return_tensors="pt",truncation=True).to("cuda")
    translated=trans_model.generate(**tokens)
    response=trans_tokenizer.decode(translated[0],skip_special_tokens=True)
    last=response
  elif input_text=="show favorites":
    if os.path.exists(fav):
      print("    Favorite Excuses   ")
      with open(fav,"r",encoding="utf-8") as f:
        for line in f:
          print("-",line.strip())
      print()
    else:
      print("Bot: No response to save")

  elif "speak" in input_text.lower():
    if last_excuse:
        from TTS.api import TTS
        from IPython.display import Audio, display

        print("Bot: Speaking the last response...")
        if os.path.exists("output.wav"):
            os.remove("output.wav")
        if 'tts_model' not in globals():
            tts_model = TTS(model_name="tts_models/en/ljspeech/tacotron2-DDC", progress_bar=False, gpu=True)
        tts_model.tts_to_file(text=last_excuse, file_path="output.wav")
        display(Audio(filename="output.wav", autoplay=True))
    else:
        print("Bot: Nothing to speak.")


  elif "certificate" in input_text.lower() or "leave certificate" in input_text.lower():
    if "medical" in input_text.lower():
        reason = "medical leave"
    elif "travel" in input_text.lower():
        reason = "travel emergency"
    elif "family" in input_text.lower():
        reason = "family function"
    else:
        reason = "personal emergency"

    today = datetime.now().strftime("%Y-%m-%d")
    response = generate_proof_document(name="Jagath", reason=reason, date=today)
    image_path = "/content/proof.png"
    create_proof_image(response, filename=image_path)
    print("Bot: Certificate image generated.")
    print("Bot: You can manually open or download the file from the left-side Files tab in Colab.")
    last = response
    last_excuse = response


  elif "rank excuses" in input_text.lower():
    rank_excuses(history)


  elif "paraphrase" in input_text.lower():
    to_paraphrase = input_text.split(":", 1)[-1].strip()
    response = paraphrase_text(to_paraphrase)
    last = response
    last_excuse = response

  elif "emergency" in input_text.lower():
    emergency_messages = [
        " Emergency Alert: Your mother has been admitted to the hospital. Please respond immediately!",
        " Emergency Alert: There's been a fire at your residence. Please leave immediately!",
        " Emergency Alert: A family member was in an accident. Urgent attention needed!",
        " Emergency Alert: Your sibling fainted and was rushed to the ER!",
        " Emergency Alert: House break-in reported. Authorities have been informed!"
    ]
    import random
    response = random.choice(emergency_messages)
    last = response
    last_excuse = response

  elif "apology" in input_text.lower():
    if "emotional" in input_text.lower():
        prompt = "Generate an emotional guilt-tripping apology for not attending an important event"
    elif "professional" in input_text.lower():
        prompt = "Generate a professional apology email for missing a deadline"
    else:
        prompt = "Generate an apology message"

    input_ids = flan_tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")
    outputs = flan_model.generate(input_ids, min_length=20, max_length=100, do_sample=True)
    response = flan_tokenizer.decode(outputs[0], skip_special_tokens=True)
    last = response
    last_excuse = response
  elif "schedule" in input_text.lower():
    import re
    time_match = re.search(r'at (\d{1,2}:\d{2})', input_text)
    if time_match and last_excuse:
        schedule_time = time_match.group(1)
        with open("schedule.txt", "a", encoding="utf-8") as f:
            f.write(f"{schedule_time} - {last_excuse}\n")
        print(f"Bot: Excuse scheduled at {schedule_time}")
    else:
        print("Bot: Please provide time in format 'HH:MM' and make sure an excuse is generated.")


  else:
    prompt_clean = f"Generate a polite and reasonable excuse for: {input_text}"
    for i in range(5):
        input_ids = flan_tokenizer(prompt_clean, return_tensors="pt").input_ids.to("cuda")
        outputs = flan_model.generate(
            input_ids, min_length=20, max_length=100,
            do_sample=True, repetition_penalty=1.5,
            no_repeat_ngram_size=3, temperature=0.9, top_p=0.95
        )
        response = flan_tokenizer.decode(outputs[0], skip_special_tokens=True)
        if response not in used_excuses:
            last = response
            last_excuse = response
            used_excuses.append(response)
            with open("excuse_history.txt", "a", encoding="utf-8") as f:
                f.write(last_excuse + "\n")
            break


  if 'response' in locals():
    print("Bot :",response)
  else:
    print("Bot: Could not generate a response.")

/usr/local/lib/python3.11/dist-packages/TTS/api.py:70: UserWarning: `gpu` will be deprecated. Please use `tts.to(device)` instead.
  warnings.warn("`gpu` will be deprecated. Please use `tts.to(device)` instead.")


 > Downloading model to /root/.local/share/tts/tts_models--en--ljspeech--tacotron2-DDC
 > Model's license - apache 2.0
 > Check https://choosealicense.com/licenses/apache-2.0/ for more info.
 > Downloading model to /root/.local/share/tts/vocoder_models--en--ljspeech--hifigan_v2
 > Model's license - apache 2.0
 > Check https://choosealicense.com/licenses/apache-2.0/ for more info.
 > Using model: Tacotron2
 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:False
 | > num_mels:80
 | > log_func:np.log
 | > min_level_db:-100
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:20
 | > fft_size:1024
 | > power:1.5
 | > preemphasis:0.0
 | > griffin_lim_iters:60
 | > signal_norm:False
 | > symmetric_norm:True
 | > mel_fmin:0
 | > mel_fmax:8000.0
 | > pitch_fmin:1.0
 | > pitch_fmax:640.0
 | > spec_gain:1.0
 | > stft_pad_mode:reflect
 | > max_norm:4.0
 | > clip_norm:True
 | > do_trim_silence:True
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linea

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Bot: Hello! How can i help you ?


model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

You : Give an excuse for not attending family function
Bot : You are at work today. I will be back home tomorrow after finishing my shift. I did not have my tickets yet, so I am not able to come
You : Generate medical leave certificate
Proof image saved as /content/proof.png
Bot: Certificate image generated.
Bot: You can manually open or download the file from the left-side Files tab in Colab.
Bot : This is to certify that Mr./Ms. Jagath was on medical leave on 2025-08-07. Kindly grant leave accordingly.
You : Translate: I need a day off
Bot : J'ai besoin d'un jour de congé.
You : Paraphrase: i couldn't finish my assignment
Bot : Why can't I finish my assignment?
You : speak
Bot: Speaking the last response...
 > Text splitted to sentences.
["Why can't I finish my assignment?"]
 > Processing time: 1.1717569828033447
 > Real-time factor: 0.45862754669862527


Bot : Why can't I finish my assignment?
You : save favorites
Bot: Response saved to favorites
Bot : Why can't I finish my assignment?
You : show history

   history    

Bot : Why can't I finish my assignment?
You : rank excuses

Top 5 Excuses:
You are at work today. I will be back home tomorrow after finishing my shift. I did not have my tickets yet, so I am not able to come (used 1 times)

Bot : Why can't I finish my assignment?
You : bye
Bot : Good bye !


In [1]:
pip install pillow


In [1]:
pip install TTS


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 65.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 11.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 111.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 22.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 15.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 137.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.